# M0 — QKD vertical slice on the qsim kernel

Proves the **multi-rate engine** runs and produces sensible QKD physics:
`FaintPulseSource → FiberChannel → BobAMZI(+phase drift) → GatedInGaAsDetector`.

Run from the repo root (so `import qsim` resolves).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qsim.qkd.reference import build_bb84_slice, load_default_profile

prof = load_default_profile()
print(prof.name, '| eta_det =', prof.value('eta_det'), '| p_dark =', prof.value('p_dark'))

## 1) QBER & secret-key rate vs distance

In [ ]:
distances = np.arange(5, 181, 10.0)
qber, skr = [], []
for L in distances:
    rng = np.random.default_rng(1)
    _g, sched, det, _amzi = build_bb84_slice(length_km=float(L), locked=True)
    recs = sched.run(n_ticks=20, dt_slow=1e-3, pulses_per_tick=150_000, rng=rng)
    qber.append(recs[-1]['qber']); skr.append(recs[-1]['skr'])
qber, skr = np.array(qber), np.array(skr)

fig, ax1 = plt.subplots(figsize=(7,4))
ax1.plot(distances, qber*100, 'o-', color='#c0392b'); ax1.set_ylabel('QBER (%)'); ax1.set_xlabel('km')
ax1.axhline(11, ls=':', color='#c0392b', alpha=.5)
ax2 = ax1.twinx(); ax2.semilogy(distances, np.where(skr>0, skr, np.nan), 's-', color='#2471a3')
ax2.set_ylabel('SKR (bps, log)'); plt.title('QBER & SKR vs distance'); plt.show()

## 2) Phase drift over time — lock OFF vs ON (the multi-rate demo)

In [ ]:
def ts(locked):
    rng = np.random.default_rng(7)
    _g, sched, _d, _a = build_bb84_slice(length_km=25.0, locked=locked)
    r = sched.run(n_ticks=1500, dt_slow=1e-3, pulses_per_tick=60_000, rng=rng)
    return (np.array([x['t'] for x in r]), np.array([x['inst_qber'] for x in r]))

tu, qu = ts(False); tl, ql = ts(True)
plt.figure(figsize=(7,3.5))
plt.plot(tu, qu*100, color='#7f8c8d', lw=.8, label='lock OFF')
plt.plot(tl, ql*100, color='#27ae60', lw=.8, label='lock ON')
plt.xlabel('time (s)'); plt.ylabel('instantaneous QBER (%)'); plt.legend(); plt.show()
print('mean QBER  OFF=%.2f%%  ON=%.2f%%' % (qu.mean()*100, ql.mean()*100))